# 统一 Benchmark 协议（对齐 SPINRec 风格）

本笔记本先定义统一评测体系，再分步骤实现“严谨最小集”。

## Benchmark 体系大纲

1. 反事实轨迹指标（核心）
- Deletion 曲线与 AUC
- Insertion 曲线与 AUC
- NDCG@Ke 曲线
- POS@K、NEG@K、Rank@K

2. 随机与弱基线对照（必要）
- Random 排序
- 启发式排序（popularity / cosine / jaccard）
- 现有方法：SIF / Influence / SHAP

3. 稳定性与鲁棒性
- 多 seed 重复
- bootstrap 用户子集重复
- 输入微扰一致性（Kendall 或 Spearman）

4. 统计显著性（论文级）
- 每个模型每个数据集报告 mean 和 std
- 配对检验：Wilcoxon signed-rank
- 效应量：Cliff's delta

## 严谨最小集（本 notebook 已实现）
- deletion_auc_minus_random
- insertion_auc_minus_random
- POS@10
- NDCG@Ke 曲线汇总
- 3 到 5 个 seed 的 mean +/- std
- SIF vs Influence vs SHAP 的 Wilcoxon + Cliff's delta

In [4]:
# 关键超参数（前置总览单元）
KEY_BENCH_PARAMS = {
    "models": ["VAE"], # 评估模型列表，可选 "MLP", "NCF", "VAE"
    "methods": ["sif", "sif_plus", "influence", "tracin", "dvf"], # 评估方法，可选 "sif", "sif_plus", "influence", "tracin", "dvf", "shap"
    "step2_timestamp": "latest", # 训练产物时间戳目录；默认 latest，也可填如 "20260326_153000"
    "seeds": [1, 14, 53, 78, 91],
    "n_train": 192,
    "n_val": 512,
    "epochs_in_utility": 7,
    "lr_bench": 1e-3,
    "deletion_fracs": [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8],
    "insertion_fracs": [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8],
    "ndcg_ke": [5, 10, 20, 50],
    "top_k_pos": 10,
    "random_repeats": 7,
    "shap_mc_iters": 100,
    "utility_avg_repeats": 2,
    # 快速稳定模式：减少评估点数量并采用稳健基线聚合
    "fast_mode": True,
    # 曲线最多评估点数量（含首尾）；越小越快
    "max_curve_points": 4,
    # NDCG@k 最多评估点数量（含首尾）；越小越快
    "max_ndcg_points": 3,
    # 随机基线聚合方式："median" 抗噪更强，"mean" 更传统
    "random_baseline_agg": "median",
    # 是否从 step progress 断点续跑，避免重复计算
    "resume_from_progress": True,
    # 若 tracin/dvf 请求了流积分但无 checkpoint，则直接报错防止静默退化
    "strict_flow_checkpoints": True,
    # 为 True 时忽略历史 progress，强制重算（参数变更后建议开启）
    "force_recompute": True,
    # 退化缓解：checkpoint 稀疏时对轨迹做插值加密
    "ckpt_densify_points": 2,
    # 退化缓解：使用参数轨迹弧长作为离散积分权重
    "flow_use_arc_length": True,
    # 退化缓解：每个 checkpoint 对 psi 去均值，降低公共模态导致的同序退化
    "flow_center_each_checkpoint": False,
}
print("关键超参数已加载：")
for k, v in KEY_BENCH_PARAMS.items():
    print(f"- {k}: {v}")

关键超参数已加载：
- models: ['VAE']
- methods: ['sif', 'sif_plus', 'influence', 'tracin', 'dvf']
- step2_timestamp: latest
- seeds: [1, 14, 53, 78, 91]
- n_train: 192
- n_val: 512
- epochs_in_utility: 7
- lr_bench: 0.001
- deletion_fracs: [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8]
- insertion_fracs: [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8]
- ndcg_ke: [5, 10, 20, 50]
- top_k_pos: 10
- random_repeats: 7
- shap_mc_iters: 100
- utility_avg_repeats: 2
- fast_mode: True
- max_curve_points: 4
- max_ndcg_points: 3
- random_baseline_agg: median
- resume_from_progress: True
- strict_flow_checkpoints: True
- force_recompute: True
- ckpt_densify_points: 2
- flow_use_arc_length: True
- flow_center_each_checkpoint: False


In [5]:
import importlib.util
import random
import subprocess
import sys
import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

if importlib.util.find_spec("nbformat") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])
if importlib.util.find_spec("scipy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])

from scipy.stats import wilcoxon

%run ./functions.ipynb

In [6]:
def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "notebooks").exists() and (p / "requirements.txt").exists():
            return p
    return c


def _resolve_step2_dir(step2_root: Path, ts_value) -> Path:
    ts_value = str(ts_value) if ts_value is not None else "latest"
    ts_value = ts_value.strip() or "latest"

    # latest 必须优先按 latest_run.txt 指向目录解析
    if ts_value.lower() == "latest":
        latest_ptr = step2_root / "latest_run.txt"
        if latest_ptr.exists():
            ts = latest_ptr.read_text(encoding="utf-8").strip()
            p = step2_root / ts
            if p.exists() and p.is_dir():
                return p
            raise FileNotFoundError(f"latest_run.txt points to missing directory: {p}")

        # 无 latest_run.txt 时回退到最新时间戳目录
        if not step2_root.exists():
            raise FileNotFoundError(f"step2_training directory not found: {step2_root}. Please run the step2 training notebook first.")
        cands = sorted([p for p in step2_root.iterdir() if p.is_dir()], key=lambda p: p.name)
        if len(cands) > 0:
            return cands[-1]
        raise FileNotFoundError(f"No step2 run directory found under: {step2_root}")

    p = step2_root / ts_value
    if p.exists() and p.is_dir():
        return p
    raise FileNotFoundError(f"Specified step2 timestamp directory does not exist: {p}")


def _load_checkpoints_from_structure(step2_dir: Path):
    # 先尝试 all_checkpoints.pt
    ck_map = {}
    all_ck_path = step2_dir / "all_checkpoints.pt"
    if all_ck_path.exists():
        raw = torch.load(all_ck_path, map_location="cpu")
        if isinstance(raw, dict):
            for k, v in raw.items():
                ck_map[str(k).upper()] = v if isinstance(v, list) else []

    # 再按目录结构兜底：*_checkpoints/*.pt
    for model in ["MLP", "NCF", "VAE"]:
        folder = step2_dir / f"{model.lower()}_checkpoints"
        files = sorted(folder.glob("*.pt")) if folder.exists() else []
        if model not in ck_map or len(ck_map[model]) == 0:
            ck_map[model] = [str(p) for p in files]

    return ck_map


REPO_ROOT = _find_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
OUT_DIR = REPO_ROOT / "log" / "notebook_runs"
STEP2_ROOT = OUT_DIR / "step2_training"
STEP2_DIR = _resolve_step2_dir(STEP2_ROOT, KEY_BENCH_PARAMS.get("step2_timestamp", "latest"))

BENCH_DIR = OUT_DIR / "step5_unified_benchmark"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = _load_checkpoints_from_structure(STEP2_DIR)

def _normalize_state_dict(obj):
    if isinstance(obj, dict):
        if all(isinstance(k, str) for k in obj.keys()) and any(isinstance(v, torch.Tensor) for v in obj.values()):
            return obj
        for k in ["state_dict", "model_state_dict", "model", "weights"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
    raise ValueError("Cannot normalize checkpoint into a state_dict")


def _build_model_for_state(name: str, state_dict: dict):
    name_u = str(name).upper()
    state_dict = _normalize_state_dict(state_dict)

    if name_u == "MLP":
        has_backbone = any(str(k).startswith("backbone.") for k in state_dict.keys())
        if has_backbone:
            class MLPRecBackbone(nn.Module):
                def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim, hidden_dims, dropout=0.1):
                    super().__init__()
                    self.enc = SparseEncoder(sparse_fields, num_buckets, emb_dim)
                    in_dim = dense_dim + emb_dim * len(sparse_fields)
                    layers = []
                    cur = in_dim
                    for i, h in enumerate(hidden_dims):
                        layers.append(nn.Linear(cur, h))
                        layers.append(nn.ReLU())
                        if i < len(hidden_dims) - 1:
                            layers.append(nn.Dropout(dropout))
                        cur = h
                    self.backbone = nn.Sequential(*layers)
                    self.out = nn.Linear(cur, 1)

                def forward(self, dense, sparse):
                    x = torch.cat([dense, self.enc(sparse)], dim=1)
                    h = self.backbone(x)
                    return self.out(h).squeeze(1)

            emb_key = next((k for k in state_dict.keys() if str(k).startswith("enc.emb.") and str(k).endswith(".weight")), None)
            emb_dim = int(state_dict[emb_key].shape[1]) if emb_key is not None else 16
            dense_dim = int(prepared.dense.shape[1])
            in_dim = dense_dim + emb_dim * len(prepared.sparse_fields)
            hidden_dims = []
            idx = 0
            while f"backbone.{idx}.weight" in state_dict:
                w = state_dict[f"backbone.{idx}.weight"]
                if int(w.shape[1]) == in_dim or len(hidden_dims) > 0:
                    hidden_dims.append(int(w.shape[0]))
                idx += 3
            hidden_dims = hidden_dims if len(hidden_dims) > 0 else [128, 64]
            return MLPRecBackbone(prepared.sparse_fields, dense_dim, NUM_BUCKETS, emb_dim=emb_dim, hidden_dims=hidden_dims, dropout=0.1)

        emb_key = next((k for k in state_dict.keys() if str(k).startswith("enc.emb.") and str(k).endswith(".weight")), None)
        emb_dim = int(state_dict[emb_key].shape[1]) if emb_key is not None else 16
        return MLPRec(prepared.sparse_fields, int(prepared.dense.shape[1]), NUM_BUCKETS, emb_dim=emb_dim)

    if name_u == "NCF":
        emb_dim = int(state_dict.get("ug.weight", torch.empty(0, 32)).shape[1]) if "ug.weight" in state_dict else 32

        linear_idxs = []
        for k in state_dict.keys():
            sk = str(k)
            if sk.startswith("mlp.") and sk.endswith(".weight"):
                parts = sk.split(".")
                if len(parts) >= 3 and parts[1].isdigit():
                    linear_idxs.append(int(parts[1]))
        linear_idxs = sorted(set(linear_idxs))

        if len(linear_idxs) == 0:
            return NCFRec(NUM_BUCKETS, emb_dim=emb_dim)

        mlp_hidden_dims = [int(state_dict[f"mlp.{i}.weight"].shape[0]) for i in linear_idxs if f"mlp.{i}.weight" in state_dict]
        if len(mlp_hidden_dims) == 0:
            return NCFRec(NUM_BUCKETS, emb_dim=emb_dim)

        class NCFRecStateCompat(nn.Module):
            def __init__(self, num_buckets, emb_dim, mlp_hidden_dims, dropout=0.1):
                super().__init__()
                e = int(emb_dim)
                self.ug = nn.Embedding(num_buckets, e)
                self.ig = nn.Embedding(num_buckets, e)
                self.um = nn.Embedding(num_buckets, e)
                self.im = nn.Embedding(num_buckets, e)

                layers = []
                in_dim = e * 2
                hs = [int(h) for h in mlp_hidden_dims]
                for i, h in enumerate(hs):
                    layers.append(nn.Linear(in_dim, h))
                    layers.append(nn.ReLU())
                    if i < len(hs) - 1:
                        layers.append(nn.Dropout(float(dropout)))
                    in_dim = h
                self.mlp = nn.Sequential(*layers)
                self.out = nn.Linear(e + hs[-1], 1)

            def forward(self, dense, sparse):
                u = sparse["user_id"]
                i = sparse["item_id"]
                gmf = self.ug(u) * self.ig(i)
                m = self.mlp(torch.cat([self.um(u), self.im(i)], dim=1))
                return self.out(torch.cat([gmf, m], dim=1)).squeeze(1)

        return NCFRecStateCompat(NUM_BUCKETS, emb_dim=emb_dim, mlp_hidden_dims=mlp_hidden_dims, dropout=0.1)

    if name_u == "VAE":
        emb_key = next((k for k in state_dict.keys() if str(k).startswith("enc_sparse.emb.") and str(k).endswith(".weight")), None)
        emb_dim = int(state_dict[emb_key].shape[1]) if emb_key is not None else 16

        enc_linear_idxs = []
        dec_linear_idxs = []
        for k in state_dict.keys():
            sk = str(k)
            if sk.startswith("encoder.") and sk.endswith(".weight"):
                parts = sk.split(".")
                if len(parts) >= 3 and parts[1].isdigit():
                    enc_linear_idxs.append(int(parts[1]))
            if sk.startswith("decoder.") and sk.endswith(".weight"):
                parts = sk.split(".")
                if len(parts) >= 3 and parts[1].isdigit():
                    dec_linear_idxs.append(int(parts[1]))

        enc_linear_idxs = sorted(set(enc_linear_idxs))
        dec_linear_idxs = sorted(set(dec_linear_idxs))

        if len(enc_linear_idxs) == 0 or len(dec_linear_idxs) == 0:
            latent_dim = int(state_dict["mu.weight"].shape[0]) if "mu.weight" in state_dict else 32
            return VAERec(prepared.sparse_fields, int(prepared.dense.shape[1]), NUM_BUCKETS, emb_dim=emb_dim, latent_dim=latent_dim)

        class VAERecStateCompat(nn.Module):
            def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim, state_dict, enc_idxs, dec_idxs, dropout=0.1):
                super().__init__()
                self.enc_sparse = SparseEncoder(sparse_fields, num_buckets, int(emb_dim))

                max_enc = int(max(enc_idxs))
                enc_modules = []
                for i in range(max_enc + 1):
                    if i in enc_idxs and f"encoder.{i}.weight" in state_dict:
                        w = state_dict[f"encoder.{i}.weight"]
                        enc_modules.append(nn.Linear(int(w.shape[1]), int(w.shape[0])))
                    else:
                        if i % 3 == 1:
                            enc_modules.append(nn.ReLU())
                        elif i % 3 == 2:
                            enc_modules.append(nn.Dropout(float(dropout)))
                        else:
                            enc_modules.append(nn.Identity())
                self.encoder = nn.Sequential(*enc_modules)

                latent_dim = int(state_dict["mu.weight"].shape[0]) if "mu.weight" in state_dict else int(state_dict[f"encoder.{enc_idxs[-1]}.weight"].shape[0])
                self.mu = nn.Linear(int(state_dict["mu.weight"].shape[1]), int(state_dict["mu.weight"].shape[0])) if "mu.weight" in state_dict else nn.Linear(latent_dim, latent_dim)
                self.logvar = nn.Linear(int(state_dict["logvar.weight"].shape[1]), int(state_dict["logvar.weight"].shape[0])) if "logvar.weight" in state_dict else nn.Linear(latent_dim, latent_dim)

                max_dec = int(max(dec_idxs))
                dec_modules = []
                for i in range(max_dec + 1):
                    if i in dec_idxs and f"decoder.{i}.weight" in state_dict:
                        w = state_dict[f"decoder.{i}.weight"]
                        dec_modules.append(nn.Linear(int(w.shape[1]), int(w.shape[0])))
                    else:
                        if i % 3 == 1:
                            dec_modules.append(nn.ReLU())
                        elif i % 3 == 2:
                            dec_modules.append(nn.Dropout(float(dropout)))
                        else:
                            dec_modules.append(nn.Identity())
                self.decoder = nn.Sequential(*dec_modules)

                if "out.weight" in state_dict:
                    self.out = nn.Linear(int(state_dict["out.weight"].shape[1]), int(state_dict["out.weight"].shape[0]))
                else:
                    last_dec = int(state_dict[f"decoder.{dec_idxs[-1]}.weight"].shape[0])
                    self.out = nn.Linear(last_dec, 1)

            def _reparameterize(self, mu, logvar):
                if self.training:
                    std = torch.exp(0.5 * logvar)
                    eps = torch.randn_like(std)
                    return mu + eps * std
                return mu

            def forward(self, dense, sparse):
                x = torch.cat([dense, self.enc_sparse(sparse)], dim=1)
                h = self.encoder(x)
                mu = self.mu(h)
                logvar = self.logvar(h)
                z = self._reparameterize(mu, logvar)
                d = self.decoder(z)
                return self.out(d).squeeze(1)

        return VAERecStateCompat(prepared.sparse_fields, int(prepared.dense.shape[1]), NUM_BUCKETS, emb_dim=emb_dim, state_dict=state_dict, enc_idxs=enc_linear_idxs, dec_idxs=dec_linear_idxs, dropout=0.1)

    return build_model(name=name_u, sparse_fields=prepared.sparse_fields, dense_dim=int(prepared.dense.shape[1]), num_buckets=NUM_BUCKETS)

cache_path = REPO_ROOT / "log" / "notebook_cache" / "prepared_data.pt"
cache = torch.load(cache_path, map_location="cpu", weights_only=False)
prepared = PreparedData(
    dense=cache["dense"],
    sparse=cache["sparse"],
    labels=cache["labels"],
    sample_ids=cache["sample_ids"],
    sparse_fields=cache["sparse_fields"],
    dense_fields=cache["dense_fields"],
)

NUM_BUCKETS = int(cache["num_buckets"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(42)

print("REPO_ROOT:", REPO_ROOT)
print("STEP2_ROOT:", STEP2_ROOT)
print("STEP2_DIR selected:", STEP2_DIR)
print("DEVICE:", DEVICE)
print("Prepared samples:", int(prepared.labels.shape[0]))
print("checkpoint groups loaded:", {k: len(v) for k, v in CHECKPOINTS.items()})


REPO_ROOT: D:\挑战杯\RecAcc\RecAcc
STEP2_ROOT: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step2_training
STEP2_DIR selected: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step2_training\20260420_162858
DEVICE: cuda
Prepared samples: 777818
checkpoint groups loaded: {'MLP': 9, 'VAE': 10, 'NCF': 7}


In [7]:
# 
# Step 1: 统一配置（从前置关键超参数单元读取）
BENCH_MODELS = list(KEY_BENCH_PARAMS["models"])
ATTR_METHODS = list(KEY_BENCH_PARAMS["methods"])
SEEDS = list(KEY_BENCH_PARAMS["seeds"])
N_TRAIN = int(KEY_BENCH_PARAMS["n_train"])
N_VAL = int(KEY_BENCH_PARAMS["n_val"])
EPOCHS_IN_UTILITY = int(KEY_BENCH_PARAMS["epochs_in_utility"])
LR_BENCH = float(KEY_BENCH_PARAMS["lr_bench"])
DEL_FRACS = list(KEY_BENCH_PARAMS["deletion_fracs"])
INS_FRACS = list(KEY_BENCH_PARAMS["insertion_fracs"])
NDCG_KE = list(KEY_BENCH_PARAMS["ndcg_ke"])
TOP_K_POS = int(KEY_BENCH_PARAMS["top_k_pos"])
RANDOM_REPEATS = int(KEY_BENCH_PARAMS["random_repeats"])
SHAP_ESTIMATOR = "permutation"
SHAP_MC_ITERS = int(KEY_BENCH_PARAMS["shap_mc_iters"])
SHAP_CACHE_MAX = 4096
UTILITY_AVG_REPEATS = int(KEY_BENCH_PARAMS["utility_avg_repeats"])
FAST_MODE = bool(KEY_BENCH_PARAMS.get("fast_mode", False))
MAX_CURVE_POINTS = int(KEY_BENCH_PARAMS.get("max_curve_points", 0))
MAX_NDCG_POINTS = int(KEY_BENCH_PARAMS.get("max_ndcg_points", 0))
RANDOM_BASELINE_AGG = str(KEY_BENCH_PARAMS.get("random_baseline_agg", "median")).lower()
RESUME_FROM_PROGRESS = bool(KEY_BENCH_PARAMS.get("resume_from_progress", True))
STRICT_FLOW_CHECKPOINTS = bool(KEY_BENCH_PARAMS.get("strict_flow_checkpoints", True))
FORCE_RECOMPUTE = bool(KEY_BENCH_PARAMS.get("force_recompute", False))
UTILITY_SHUFFLE = False
UTILITY_USE_SUBSET_SEED = True
CKPT_DENSIFY_POINTS = int(KEY_BENCH_PARAMS.get("ckpt_densify_points", 0))
FLOW_USE_ARC_LENGTH = bool(KEY_BENCH_PARAMS.get("flow_use_arc_length", True))
FLOW_CENTER_EACH_CHECKPOINT = bool(KEY_BENCH_PARAMS.get("flow_center_each_checkpoint", True))

print("配置已载入。")
print("bench_models:", BENCH_MODELS)
print("attr_methods:", ATTR_METHODS)
print("ckpt_densify_points:", CKPT_DENSIFY_POINTS)
print("flow_use_arc_length:", FLOW_USE_ARC_LENGTH)
print("flow_center_each_checkpoint:", FLOW_CENTER_EACH_CHECKPOINT)
print("fast_mode:", FAST_MODE)
print("max_curve_points:", MAX_CURVE_POINTS)
print("max_ndcg_points:", MAX_NDCG_POINTS)
print("random_baseline_agg:", RANDOM_BASELINE_AGG)
print("resume_from_progress:", RESUME_FROM_PROGRESS)
print("strict_flow_checkpoints:", STRICT_FLOW_CHECKPOINTS)
print("force_recompute:", FORCE_RECOMPUTE)



配置已载入。
bench_models: ['VAE']
attr_methods: ['sif', 'sif_plus', 'influence', 'tracin', 'dvf']
ckpt_densify_points: 2
flow_use_arc_length: True
flow_center_each_checkpoint: False
fast_mode: True
max_curve_points: 4
max_ndcg_points: 3
random_baseline_agg: median
resume_from_progress: True
strict_flow_checkpoints: True
force_recompute: True


In [8]:
def _collect_from_loader(loader, n_samples: int):
    dense_list = []
    y_list = []
    sid_list = []
    sparse_list = {k: [] for k in prepared.sparse_fields}
    c = 0
    for batch in loader:
        dense, sparse, y, sid = _unpack_batch(batch)
        bs = int(dense.shape[0])
        take = min(bs, n_samples - c)
        if take <= 0:
            break
        dense_list.append(dense[:take].detach().cpu())
        y_list.append(y[:take].detach().cpu())
        sid_list.append(sid[:take].detach().cpu())
        for k in prepared.sparse_fields:
            sparse_list[k].append(sparse[k][:take].detach().cpu())
        c += take
        if c >= n_samples:
            break


    dense = torch.cat(dense_list, dim=0)
    labels = torch.cat(y_list, dim=0)
    sample_ids = torch.cat(sid_list, dim=0)
    sparse = {k: torch.cat(v, dim=0) for k, v in sparse_list.items()}
    return PreparedData(
        dense=dense,
        sparse=sparse,
        labels=labels,
        sample_ids=sample_ids,
        sparse_fields=prepared.sparse_fields,
        dense_fields=prepared.dense_fields,
    )



def _loader_from_prepared(p: PreparedData, batch_size=32, shuffle=False):
    ds = RecDataset(p)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)



def _subset_prepared(p: PreparedData, idxs):
    idxs = torch.tensor(list(idxs), dtype=torch.long)
    return PreparedData(
        dense=p.dense[idxs],
        sparse={k: v[idxs] for k, v in p.sparse.items()},
        labels=p.labels[idxs],
        sample_ids=p.sample_ids[idxs],
        sparse_fields=p.sparse_fields,
        dense_fields=p.dense_fields,
    )



def _rankdata(x):
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        rank = (i + j + 2) / 2.0
        ranks[order[i : j + 1]] = rank
        i = j + 1
    return ranks



def _spearman(a, b):
    ra = _rankdata(a)
    rb = _rankdata(b)
    if np.std(ra) == 0 or np.std(rb) == 0:
        return 0.0
    return float(np.corrcoef(ra, rb)[0, 1])



def _cliffs_delta(x, y):
    x = [float(v) for v in x]
    y = [float(v) for v in y]
    gt = 0
    lt = 0
    for a in x:
        for b in y:
            if a > b:
                gt += 1
            elif a < b:
                lt += 1
    n = max(len(x) * len(y), 1)
    return float((gt - lt) / n)

In [ ]:
def _build_method_scores(model_name, model_base, train_small_loader, val_small_loader, ids, mc_iters):
    scores = {}

    def _load_state_obj(ck_obj):
        if isinstance(ck_obj, str):
            return _normalize_state_dict(torch.load(ck_obj, map_location="cpu"))
        return _normalize_state_dict({k: v.detach().cpu().clone() for k, v in ck_obj.items()})

    def _interp_state(sa, sb, alpha: float):
        out = {}
        for k in sa.keys():
            ta = sa[k]
            tb = sb[k]
            if torch.is_floating_point(ta) and torch.is_floating_point(tb):
                out[k] = ((1.0 - alpha) * ta + alpha * tb).detach().cpu()
            else:
                out[k] = ta.detach().cpu().clone() if alpha < 0.5 else tb.detach().cpu().clone()
        return out

    def _state_distance(sa, sb):
        s = 0.0
        for k in sa.keys():
            da = sa[k].detach().float().view(-1)
            db = sb[k].detach().float().view(-1)
            diff = da - db
            s += float(torch.dot(diff, diff).item())
        return float(np.sqrt(max(s, 0.0)))

    def _densify_states(states, n_interp: int):
        if n_interp <= 0 or len(states) <= 1:
            return states
        dense = [states[0]]
        for j in range(1, len(states)):
            s0, s1 = states[j - 1], states[j]
            for t in range(1, n_interp + 1):
                a = float(t / (n_interp + 1))
                dense.append(_interp_state(s0, s1, a))
            dense.append(s1)
        return dense

    def _collect_sample_grads(model_obj):
        all_grads = {}
        for batch in train_small_loader:
            dense, sparse, y, sid = _unpack_batch(batch)
            bs = dense.shape[0]
            for i in range(bs):
                sid_i = int(sid[i].item())
                if sid_i not in ids:
                    continue
                gi = sample_grad(
                    model_obj,
                    dense[i : i + 1],
                    {k: v[i : i + 1] for k, v in sparse.items()},
                    y[i : i + 1],
                    device=DEVICE,
                )
                all_grads[sid_i] = gi
        return all_grads

    if "sif" in ATTR_METHODS:
        t0 = time.perf_counter()
        sif_scores = compute_sif(
            model=model_base,
            val_loader=val_small_loader,
            train_loader_eval=train_small_loader,
            max_samples=len(ids),
            device=DEVICE,
        )
        scores["sif"] = {
            "scores": sif_scores,
            "time_sec": float(time.perf_counter() - t0),
        }

    if "sif_plus" in ATTR_METHODS:
        t0 = time.perf_counter()
        vg = val_grad(model_base, val_small_loader, device=DEVICE)
        all_grads = _collect_sample_grads(model_base)

        damp = 1e-3
        diag = None
        for sid in ids:
            g2 = all_grads[sid] * all_grads[sid]
            diag = g2 if diag is None else (diag + g2)
        diag = diag / max(len(ids), 1) + damp

        sif_plus_scores = {}
        for sid in ids:
            gi = all_grads[sid]
            sif_plus_scores[sid] = float((vg * gi / diag).sum().item())

        scores["sif_plus"] = {
            "scores": sif_plus_scores,
            "time_sec": float(time.perf_counter() - t0),
        }

    if "influence" in ATTR_METHODS:
        t0 = time.perf_counter()
        vg = val_grad(model_base, val_small_loader, device=DEVICE)
        all_grads = _collect_sample_grads(model_base)

        def hessian_vector_product(v, model, train_loader, device):
            """计算 Hessian-vector product (Hv)"""
            params = [p for p in model.parameters() if p.requires_grad]
            
            # 先计算梯度
            total_grad = None
            for batch in train_loader:
                dense, sparse, y, _ = _unpack_batch(batch)
                dense, y = dense.to(device), y.to(device)
                sparse = {k: v.to(device) for k, v in sparse.items()}
                
                pred = model(dense, sparse)
                loss = F.binary_cross_entropy_with_logits(pred, y)
                
                grads = torch.autograd.grad(loss, params, create_graph=True)
                grad_vec = torch.cat([g.view(-1) for g in grads])
                
                if total_grad is None:
                    total_grad = grad_vec
                else:
                    total_grad = grad_vec + total_grad
            
            total_grad = total_grad / len(train_loader)
            
            # 计算 Hv
            hv = torch.autograd.grad(total_grad, params, 
                                    grad_outputs=v, retain_graph=True)
            return torch.cat([h.view(-1) for h in hv])
        
        def conjugate_gradient(Hv_func, v, max_iter=100, tol=1e-5):
            """共轭梯度法求解 H^{-1} v"""
            x = torch.zeros_like(v)
            r = v.clone()
            p = v.clone()
            rs_old = torch.dot(r, r)
            
            for i in range(max_iter):
                Hp = Hv_func(p)
                alpha = rs_old / torch.dot(p, Hp)
                x += alpha * p
                r -= alpha * Hp
                rs_new = torch.dot(r, r)
                
                if rs_new < tol:
                    break
                
                beta = rs_new / rs_old
                p = r + beta * p
                rs_old = rs_new
            
            return x
        
        # 计算 H^{-1} vg
        def h_inv_vg(model, train_loader, vg, device):
            """计算 Hessian^{-1} * vg"""
            params = [p for p in model.parameters() if p.requires_grad]
            vg_vec = torch.cat([g.view(-1) for g in vg])
            
            def Hv_func(v):
                return hessian_vector_product(v, model, train_loader, device)
            
            return conjugate_gradient(Hv_func, vg_vec)
        
        # 计算精确 influence
        h_inv_vg_vec = h_inv_vg(model_base, train_small_loader, vg, DEVICE)
        
        # 将向量拆回参数结构
        def vec_to_params(vec, model):
            params = []
            idx = 0
            for p in model.parameters():
                n = p.numel()
                params.append(vec[idx:idx+n].view(p.shape))
                idx += n
            return params
        
        h_inv_vg_params = vec_to_params(h_inv_vg_vec, model_base)
    
        if_scores = {}
        for sid in ids:
            gi = all_grads[sid]
            # influence = -∇L_val^T H^{-1} ∇L_train
            influence_val = 0
            for g_param, hv_param in zip(gi, h_inv_vg_params):
                influence_val -= torch.sum(g_param * hv_param).item()
            if_scores[sid] = float(influence_val)
        
        scores["influence"] = {
            "scores": if_scores,
            "time_sec": float(time.perf_counter() - t0),
        }

    if ("tracin" in ATTR_METHODS) or ("dvf" in ATTR_METHODS):
        t0 = time.perf_counter()
        ck_list = CHECKPOINTS.get(model_name, []) if isinstance(CHECKPOINTS, dict) else []
        if len(ck_list) == 0:
            states = [{k: v.detach().cpu().clone() for k, v in model_base.state_dict().items()}]
        else:
            states = [_load_state_obj(ck) for ck in ck_list]

        states = _densify_states(states, CKPT_DENSIFY_POINTS)

        psi_nodes = []
        for st in states:
            model_ck = _build_model_for_state(model_name, st)
            model_ck.load_state_dict(st)
            model_ck.to(DEVICE)

            vg_ck = val_grad(model_ck, val_small_loader, device=DEVICE)
            grads_ck = _collect_sample_grads(model_ck)

            psi = {sid: float((vg_ck * grads_ck[sid]).sum().item()) for sid in ids}
            if FLOW_CENTER_EACH_CHECKPOINT:
                mpsi = float(np.mean(list(psi.values())))
                psi = {sid: float(v - mpsi) for sid, v in psi.items()}
            psi_nodes.append(psi)

        if len(states) <= 1:
            seg_w = []
            node_w = [1.0]
        else:
            seg_w = []
            for j in range(len(states) - 1):
                if FLOW_USE_ARC_LENGTH:
                    w = _state_distance(states[j], states[j + 1])
                else:
                    w = 1.0
                seg_w.append(max(float(w), 1e-8))
            sw = float(sum(seg_w))
            if sw > 0:
                seg_w = [w * (len(seg_w) / sw) for w in seg_w]

            node_w = [0.0 for _ in range(len(states))]
            node_w[0] = seg_w[0]
            node_w[-1] = seg_w[-1]
            for j in range(1, len(states) - 1):
                node_w[j] = 0.5 * (seg_w[j - 1] + seg_w[j])

        if "tracin" in ATTR_METHODS:
            tracin_scores = {sid: 0.0 for sid in ids}
            for j, psi in enumerate(psi_nodes):
                w = float(node_w[j])
                for sid in ids:
                    tracin_scores[sid] += float(w * psi[sid])
            scores["tracin"] = {
                "scores": tracin_scores,
                "time_sec": float(time.perf_counter() - t0),
            }

        if "dvf" in ATTR_METHODS:
            dvf_scores = {sid: 0.0 for sid in ids}
            if len(states) <= 1:
                for sid in ids:
                    dvf_scores[sid] = float(psi_nodes[0][sid])
            else:
                for j in range(len(seg_w)):
                    w = float(seg_w[j])
                    psi_l = psi_nodes[j]
                    psi_r = psi_nodes[j + 1]
                    for sid in ids:
                        dvf_scores[sid] += float(0.5 * w * (psi_l[sid] + psi_r[sid]))
            scores["dvf"] = {
                "scores": dvf_scores,
                "time_sec": float(time.perf_counter() - t0),
            }

    if "shap" in ATTR_METHODS:
        t0 = time.perf_counter()
        shap_scores = {sid: 0.0 for sid in ids}
        scores["shap"] = {"scores": shap_scores, "time_sec": 0.0}
        scores["shap"]["_t0"] = t0

    return scores


def _curve_auc(x, y):
    return float(np.trapezoid(np.asarray(y, dtype=float), np.asarray(x, dtype=float)))


def _dcg(rels):
    rels = np.asarray(rels, dtype=float)
    if rels.size == 0:
        return 0.0
    denom = np.log2(np.arange(2, 2 + rels.size))
    return float((rels / denom).sum())

In [10]:
def _subsample_list(seq, max_points: int):
    seq = list(seq)
    if max_points <= 0 or len(seq) <= max_points:
        return seq
    idx = np.linspace(0, len(seq) - 1, num=max_points)
    idx = sorted(set(int(round(v)) for v in idx))
    if 0 not in idx:
        idx = [0] + idx
    if (len(seq) - 1) not in idx:
        idx = idx + [len(seq) - 1]
    idx = sorted(set(idx))
    return [seq[i] for i in idx]

def _norm_auc(auc_value: float, x_points):
    x_points = list(x_points)
    if len(x_points) < 2:
        return float(auc_value)
    span = float(max(x_points) - min(x_points))
    if span <= 1e-12:
        return float(auc_value)
    return float(auc_value / span)

def _run_one_setting(seed, progress_path=None, done_models=None):
    seed_everything(seed)
    random.seed(seed)
    np.random.seed(seed)

    done_models = set(done_models or [])

    train_loader, train_eval_loader, val_loader = make_loaders(
        prepared, batch_size=256, val_ratio=0.2, seed=seed
    )

    train_small = _collect_from_loader(train_eval_loader, N_TRAIN)
    val_small = _collect_from_loader(val_loader, N_VAL)
    train_small_loader = _loader_from_prepared(train_small, batch_size=16, shuffle=True)
    val_small_loader = _loader_from_prepared(val_small, batch_size=32, shuffle=False)

    ids = [int(x.item()) for x in train_small.sample_ids]
    id_to_pos = {sid: i for i, sid in enumerate(ids)}
    full_idxs = list(range(len(ids)))

    eval_del_fracs = _subsample_list(DEL_FRACS, MAX_CURVE_POINTS if FAST_MODE else 0)
    eval_ins_fracs = _subsample_list(INS_FRACS, MAX_CURVE_POINTS if FAST_MODE else 0)
    eval_ndcg_ke = _subsample_list(NDCG_KE, MAX_NDCG_POINTS if FAST_MODE else 0)

    all_rows = []
    curve_rows = []

    progress_path = Path(progress_path) if progress_path is not None else None

    for bench_model in BENCH_MODELS:
        if bench_model in done_models:
            print(f"skip existing: seed={seed}, model={bench_model}")
            continue

        print(f"\n=== seed={seed} model={bench_model} ===")
        row_start = len(all_rows)

        state_path = STEP2_DIR / f"{bench_model.lower()}_final_state.pt"
        state_obj = _normalize_state_dict(torch.load(state_path, map_location="cpu"))
        model_base = _build_model_for_state(bench_model, state_obj)
        model_base.load_state_dict(state_obj)
        model_base.to(DEVICE)

        base_state = {k: v.detach().cpu().clone() for k, v in model_base.state_dict().items()}

        def _stable_subset_seed(selected_idxs, rep):
            key = sorted([int(x) for x in selected_idxs])
            acc = 0
            for t, x in enumerate(key):
                acc = (acc + (t + 1) * (x + 17)) % 2147483647
            name_acc = sum([ord(ch) for ch in bench_model])
            return int((seed * 1000003 + acc * 1315423911 + rep * 2654435761 + name_acc) % 2147483647)

        def utility_auc(selected_idxs):
            selected_idxs = list(selected_idxs)
            if len(selected_idxs) == 0:
                return 0.5
            sub = _subset_prepared(train_small, selected_idxs)
            vals = []
            for rep in range(max(int(UTILITY_AVG_REPEATS), 1)):
                if UTILITY_USE_SUBSET_SEED:
                    s2 = _stable_subset_seed(selected_idxs, rep)
                    seed_everything(s2)
                    random.seed(s2)
                    np.random.seed(s2)

                sub_loader = _loader_from_prepared(sub, batch_size=16, shuffle=UTILITY_SHUFFLE)
                m = _build_model_for_state(bench_model, base_state)
                m.load_state_dict(base_state)
                _hist, _snap = train_model(
                    m,
                    sub_loader,
                    val_small_loader,
                    epochs=EPOCHS_IN_UTILITY,
                    lr=LR_BENCH,
                    device=DEVICE,
                    verbose=False,
                )
                vals.append(float(evaluate_model(m, val_small_loader, device=DEVICE)["auc"]))
            return float(np.mean(vals))

        @lru_cache(maxsize=SHAP_CACHE_MAX)
        def utility_auc_cached(sorted_idxs_key):
            return utility_auc(list(sorted_idxs_key))

        full_u = utility_auc_cached(tuple(sorted(full_idxs)))

        method_pack = _build_method_scores(
            model_name=bench_model,
            model_base=model_base,
            train_small_loader=train_small_loader,
            val_small_loader=val_small_loader,
            ids=ids,
            mc_iters=SHAP_MC_ITERS,
        )

        if "shap" in method_pack:
            t0 = method_pack["shap"]["_t0"]
            shap_scores = method_pack["shap"]["scores"]
            for _ in range(int(SHAP_MC_ITERS)):
                perm = full_idxs.copy()
                random.shuffle(perm)
                prefix = []
                prev_u = 0.5
                for idx in perm:
                    prefix.append(idx)
                    key = tuple(sorted(prefix))
                    cur_u = utility_auc_cached(key)
                    sid = ids[idx]
                    shap_scores[sid] += (cur_u - prev_u)
                    prev_u = cur_u
            for sid in ids:
                shap_scores[sid] /= max(int(SHAP_MC_ITERS), 1)
            method_pack["shap"]["time_sec"] = float(time.perf_counter() - t0)
            method_pack["shap"].pop("_t0", None)

        def _evaluate_one_ranking(scores, method_name, record_curve=True, compute_item_metrics=True):
            ranked = sorted(ids, key=lambda sid: float(scores.get(sid, 0.0)), reverse=True)

            deletion_curve = []
            for frac in eval_del_fracs:
                k = max(1, int(round(len(ids) * float(frac))))
                removed = set(ranked[:k])
                keep = [j for j in full_idxs if ids[j] not in removed]
                deletion_curve.append(float(full_u - utility_auc_cached(tuple(sorted(keep)))))

            insertion_curve = []
            for frac in eval_ins_fracs:
                k = max(1, int(round(len(ids) * float(frac))))
                kept = set(ranked[:k])
                keep = [j for j in full_idxs if ids[j] in kept]
                insertion_curve.append(float(utility_auc_cached(tuple(sorted(keep))) - 0.5))

            if compute_item_metrics:
                item_delta = {}
                max_ke = max(eval_ndcg_ke) if len(eval_ndcg_ke) > 0 else TOP_K_POS
                eval_top = ranked[: max(max_ke, TOP_K_POS)]
                for sid in eval_top:
                    i = id_to_pos[sid]
                    keep = [j for j in full_idxs if j != i]
                    item_delta[sid] = float(full_u - utility_auc_cached(tuple(sorted(keep))))
                pos_at_10 = float(np.mean([1.0 if item_delta[sid] > 0 else 0.0 for sid in ranked[:TOP_K_POS]]))
                ndcg_values = []
                for ke in eval_ndcg_ke:
                    top_ids = ranked[:ke]
                    rel = [max(item_delta.get(sid, 0.0), 0.0) for sid in top_ids]
                    ideal = sorted(rel, reverse=True)
                    dcg = _dcg(rel)
                    idcg = _dcg(ideal)
                    ndcg_values.append(float(dcg / idcg) if idcg > 0 else 0.0)
            else:
                pos_at_10 = float("nan")
                ndcg_values = [float("nan") for _ in eval_ndcg_ke]

            deletion_auc = _curve_auc(eval_del_fracs, deletion_curve)
            insertion_auc = _curve_auc(eval_ins_fracs, insertion_curve)
            ndcg_auc = _curve_auc(eval_ndcg_ke, ndcg_values)

            deletion_auc_norm = _norm_auc(deletion_auc, eval_del_fracs)
            insertion_auc_norm = _norm_auc(insertion_auc, eval_ins_fracs)

            if record_curve:
                curve_rows.extend([
                    {"seed": seed, "model": bench_model, "method": method_name, "curve": "deletion", "x": float(x), "y": float(y)}
                    for x, y in zip(eval_del_fracs, deletion_curve)
                ])
                curve_rows.extend([
                    {"seed": seed, "model": bench_model, "method": method_name, "curve": "insertion", "x": float(x), "y": float(y)}
                    for x, y in zip(eval_ins_fracs, insertion_curve)
                ])
                curve_rows.extend([
                    {"seed": seed, "model": bench_model, "method": method_name, "curve": "ndcg_ke", "x": float(x), "y": float(y)}
                    for x, y in zip(eval_ndcg_ke, ndcg_values)
                ])

            return {
                "deletion_auc": float(deletion_auc),
                "insertion_auc": float(insertion_auc),
                "deletion_auc_norm": float(deletion_auc_norm),
                "insertion_auc_norm": float(insertion_auc_norm),
                "ndcg_ke_auc": float(ndcg_auc),
                "pos_at_10": float(pos_at_10),
            }

        rnd_del = []
        rnd_ins = []
        rnd_del_norm = []
        rnd_ins_norm = []
        rng_random = np.random.default_rng(seed * 1009 + len(ids))
        for _ in range(max(int(RANDOM_REPEATS), 1)):
            rr = ids.copy()
            rng_random.shuffle(rr)
            rs = {sid: float(len(rr) - i) for i, sid in enumerate(rr)}
            rmet = _evaluate_one_ranking(rs, "random", record_curve=False, compute_item_metrics=False)
            rnd_del.append(float(rmet["deletion_auc"]))
            rnd_ins.append(float(rmet["insertion_auc"]))
            rnd_del_norm.append(float(rmet["deletion_auc_norm"]))
            rnd_ins_norm.append(float(rmet["insertion_auc_norm"]))

        if RANDOM_BASELINE_AGG == "mean":
            rnd_del_ref = float(np.mean(rnd_del))
            rnd_ins_ref = float(np.mean(rnd_ins))
            rnd_del_norm_ref = float(np.mean(rnd_del_norm))
            rnd_ins_norm_ref = float(np.mean(rnd_ins_norm))
        else:
            rnd_del_ref = float(np.median(rnd_del))
            rnd_ins_ref = float(np.median(rnd_ins))
            rnd_del_norm_ref = float(np.median(rnd_del_norm))
            rnd_ins_norm_ref = float(np.median(rnd_ins_norm))

        for method_name, payload in method_pack.items():
            met = _evaluate_one_ranking(payload["scores"], method_name)
            all_rows.append({
                "seed": int(seed),
                "model": bench_model,
                "method": method_name,
                "time_sec": float(payload["time_sec"]),
                "deletion_auc": float(met["deletion_auc"]),
                "insertion_auc": float(met["insertion_auc"]),
                "deletion_auc_norm": float(met["deletion_auc_norm"]),
                "insertion_auc_norm": float(met["insertion_auc_norm"]),
                "deletion_auc_minus_random": float(met["deletion_auc"] - rnd_del_ref),
                "insertion_auc_minus_random": float(met["insertion_auc"] - rnd_ins_ref),
                "deletion_auc_norm_minus_random": float(met["deletion_auc_norm"] - rnd_del_norm_ref),
                "insertion_auc_norm_minus_random": float(met["insertion_auc_norm"] - rnd_ins_norm_ref),
                "pos_at_10": float(met["pos_at_10"]),
                "ndcg_ke_auc": float(met["ndcg_ke_auc"]),
            })

        if progress_path is not None:
            step_df = pd.DataFrame(all_rows[row_start:])
            if len(step_df) > 0:
                write_header = not progress_path.exists()
                step_df.to_csv(progress_path, mode="a", header=write_header, index=False)
                print(f"step saved: seed={seed}, model={bench_model}, rows={len(step_df)} -> {progress_path}")

    return pd.DataFrame(all_rows), pd.DataFrame(curve_rows)


In [11]:
all_detail = []
all_curves = []
progress_path = BENCH_DIR / "benchmark_step_progress.csv"
done_map = {}
if FORCE_RECOMPUTE and progress_path.exists():
    progress_path.unlink()
    print(f"force_recompute enabled, removed old progress: {progress_path}")
if RESUME_FROM_PROGRESS and progress_path.exists():
    try:
        progress_df = pd.read_csv(progress_path)
        if len(progress_df) > 0:
            for sd, grp in progress_df.groupby("seed"):
                done_map[int(sd)] = set(grp["model"].dropna().astype(str).tolist())
            all_detail.append(progress_df)
            print(f"resume enabled: loaded {len(progress_df)} existing rows from {progress_path}")
    except Exception as e:
        print("resume parse failed, fallback to fresh run:", e)
        if progress_path.exists():
            progress_path.unlink()
else:
    if progress_path.exists() and not RESUME_FROM_PROGRESS:
        progress_path.unlink()
for sd in SEEDS:
    done_models = done_map.get(int(sd), set()) if RESUME_FROM_PROGRESS else set()
    df_seed, df_curve = _run_one_setting(sd, progress_path=progress_path, done_models=done_models)
    if len(df_seed) > 0:
        all_detail.append(df_seed)
    if len(df_curve) > 0:
        all_curves.append(df_curve)
detail_df = pd.concat(all_detail, ignore_index=True) if all_detail else pd.DataFrame()
if len(detail_df) > 0:
    detail_df = detail_df.drop_duplicates(subset=["seed", "model", "method"], keep="last").reset_index(drop=True)
curve_df = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
detail_path = BENCH_DIR / "benchmark_detail.csv"
curve_path = BENCH_DIR / "benchmark_curves.csv"
save_table(detail_df, str(detail_path))
save_table(curve_df, str(curve_path))
print("step-level progress csv:", progress_path)
detail_df.head()



=== seed=1 model=VAE ===


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


step saved: seed=1, model=VAE, rows=5 -> D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_step_progress.csv

=== seed=14 model=VAE ===


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


step saved: seed=14, model=VAE, rows=5 -> D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_step_progress.csv

=== seed=53 model=VAE ===


...

c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


step saved: seed=78, model=VAE, rows=5 -> D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_step_progress.csv

=== seed=91 model=VAE ===


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


step saved: seed=91, model=VAE, rows=5 -> D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_step_progress.csv
saved: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_detail.csv
saved: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_curves.csv
step-level progress csv: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_step_progress.csv


,seed,model,method,time_sec,deletion_auc,insertion_auc,deletion_auc_norm,insertion_auc_norm,deletion_auc_minus_random,insertion_auc_minus_random,deletion_auc_norm_minus_random,insertion_auc_norm_minus_random,pos_at_10,ndcg_ke_auc
0,1,VAE,sif,4.003625,-0.005152,0.055493,-0.006870,0.073991,-0.011628,-0.091967,-0.015504,-0.122623,0.6,29.440920
1,1,VAE,sif_plus,7.379697,0.004423,0.009700,0.005897,0.012933,-0.002053,-0.137761,-0.002737,-0.183681,0.8,30.694239
2,1,VAE,influence,7.170310,0.096726,0.133330,0.128968,0.177773,0.090250,-0.014130,0.120334,-0.018841,1.0,39.088405
3,1,VAE,tracin,199.009612,0.009454,-0.021440,0.012606,-0.028587,0.002979,-0.168901,0.003972,-0.225201,0.8,31.257062
4,1,VAE,dvf,199.013651,0.009454,-0.021440,0.012606,-0.028587,0.002979,-0.168901,0.003972,-0.225201,0.8,30.928065


In [12]:
if len(detail_df) == 0:
    summary_df = pd.DataFrame()
    stats_df = pd.DataFrame()
else:
    agg_cols = [
        "deletion_auc",
        "insertion_auc",
        "deletion_auc_norm",
        "insertion_auc_norm",
        "deletion_auc_minus_random",
        "insertion_auc_minus_random",
        "deletion_auc_norm_minus_random",
        "insertion_auc_norm_minus_random",
        "pos_at_10",
        "ndcg_ke_auc",
        "time_sec",
    ]
    group_cols = ["model", "method"]
    mean_std = detail_df.groupby(group_cols, as_index=False)[agg_cols].agg(["mean", "std"]).reset_index()
    mean_std.columns = ["_".join([c for c in col if c]).strip("_") for col in mean_std.columns.values]

    med = detail_df.groupby(group_cols, as_index=False)[agg_cols].median().rename(columns={c: f"{c}_median" for c in agg_cols})
    q25 = detail_df.groupby(group_cols, as_index=False)[agg_cols].quantile(0.25).rename(columns={c: f"{c}_q25" for c in agg_cols})
    q75 = detail_df.groupby(group_cols, as_index=False)[agg_cols].quantile(0.75).rename(columns={c: f"{c}_q75" for c in agg_cols})
    summary_df = mean_std.merge(med, on=group_cols, how="left").merge(q25, on=group_cols, how="left").merge(q75, on=group_cols, how="left")

    metrics = ["deletion_auc_norm_minus_random", "insertion_auc_norm_minus_random", "pos_at_10", "ndcg_ke_auc"]
    stat_rows = []
    for model_name in sorted(detail_df["model"].unique().tolist()):
        dm = detail_df[detail_df["model"] == model_name].copy()
        methods = sorted(dm["method"].dropna().unique().tolist())
        pairs = []
        for i in range(len(methods)):
            for j in range(i + 1, len(methods)):
                pairs.append((methods[i], methods[j]))

        for m1, m2 in pairs:
            d1 = dm[dm["method"] == m1].sort_values("seed")
            d2 = dm[dm["method"] == m2].sort_values("seed")
            shared = sorted(set(d1["seed"].tolist()) & set(d2["seed"].tolist()))
            if len(shared) < 2:
                continue
            a = d1[d1["seed"].isin(shared)].sort_values("seed")
            b = d2[d2["seed"].isin(shared)].sort_values("seed")

            for met in metrics:
                xa = a[met].to_numpy(dtype=float)
                xb = b[met].to_numpy(dtype=float)
                try:
                    w = wilcoxon(xa, xb, zero_method="wilcox", alternative="two-sided")
                    pval = float(w.pvalue)
                    stat = float(w.statistic)
                except Exception:
                    pval = float("nan")
                    stat = float("nan")

                stat_rows.append({
                    "model": model_name,
                    "metric": met,
                    "method_a": m1,
                    "method_b": m2,
                    "wilcoxon_stat": stat,
                    "wilcoxon_p": pval,
                    "cliffs_delta": _cliffs_delta(xa, xb),
                    "n_pairs": int(len(shared)),
                })

    stats_df = pd.DataFrame(stat_rows)
    if len(stats_df) > 0:
        stats_df["wilcoxon_p_adj_holm"] = np.nan
        for (_model_name, _metric_name), grp in stats_df.groupby(["model", "metric"]):
            idx = grp.index.to_list()
            pvals = stats_df.loc[idx, "wilcoxon_p"].to_numpy(dtype=float)
            valid_pos = np.where(~np.isnan(pvals))[0]
            m = int(valid_pos.size)
            if m == 0:
                continue

            ord_pos = valid_pos[np.argsort(pvals[valid_pos])]
            p_sorted = pvals[ord_pos]
            adj_sorted = np.array([(m - i) * p_sorted[i] for i in range(m)], dtype=float)
            adj_sorted = np.maximum.accumulate(adj_sorted)
            adj_sorted = np.clip(adj_sorted, 0.0, 1.0)
            for i in range(m):
                stats_df.loc[idx[ord_pos[i]], "wilcoxon_p_adj_holm"] = float(adj_sorted[i])

summary_path = BENCH_DIR / "benchmark_summary_mean_std.csv"
stats_path = BENCH_DIR / "benchmark_significance.csv"
save_table(summary_df, str(summary_path))
save_table(stats_df, str(stats_path))
print("Saved summary:", summary_path)
print("Saved significance:", stats_path)
summary_df.head()


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


saved: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_summary_mean_std.csv
saved: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_significance.csv
Saved summary: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_summary_mean_std.csv
Saved significance: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_significance.csv


,index,model,method,deletion_auc_mean,deletion_auc_std,insertion_auc_mean,insertion_auc_std,deletion_auc_norm_mean,deletion_auc_norm_std,insertion_auc_norm_mean,...,insertion_auc_q75,deletion_auc_norm_q75,insertion_auc_norm_q75,deletion_auc_minus_random_q75,insertion_auc_minus_random_q75,deletion_auc_norm_minus_random_q75,insertion_auc_norm_minus_random_q75,pos_at_10_q75,ndcg_ke_auc_q75,time_sec_q75
0,0,VAE,dvf,0.002515,0.020086,0.005094,0.033417,0.003353,0.026781,0.006792,...,0.016876,0.012606,0.022501,0.002979,-0.152132,0.003972,-0.202843,0.8,30.928065,199.013651
1,1,VAE,influence,0.083420,0.016858,0.148041,0.011843,0.111227,0.022477,0.197388,...,0.155155,0.128968,0.206873,0.090250,-0.007347,0.120334,-0.009796,1.0,38.380772,7.170310
2,2,VAE,sif,0.001674,0.012384,0.066600,0.044145,0.002232,0.016512,0.088801,...,0.084959,0.016062,0.113279,0.009907,-0.078390,0.013209,-0.104521,0.6,29.440920,3.954798
3,3,VAE,sif_plus,0.001896,0.011227,0.034457,0.043787,0.002527,0.014969,0.045942,...,0.068114,0.012058,0.090819,0.005124,-0.100894,0.006831,-0.134525,0.8,30.694239,6.696978
4,4,VAE,tracin,0.002301,0.020283,0.004845,0.033631,0.003068,0.027044,0.006460,...,0.016876,0.012606,0.022501,0.002979,-0.152132,0.003972,-0.202843,0.8,31.257062,199.009612


In [13]:
if len(curve_df) > 0:
    display(curve_df.groupby(["curve", "method"]).agg(y_mean=("y", "mean"), y_std=("y", "std")).reset_index())

if len(stats_df) > 0:
    sort_cols = ["model", "metric", "wilcoxon_p_adj_holm"] if "wilcoxon_p_adj_holm" in stats_df.columns else ["model", "metric", "wilcoxon_p"]
    display(stats_df.sort_values(sort_cols))


,curve,method,y_mean,y_std
0,deletion,dvf,0.002264,0.029373
1,deletion,influence,0.098680,0.130418
2,deletion,sif,0.001946,0.022274
3,deletion,sif_plus,0.002821,0.017949
4,deletion,tracin,0.002086,0.029555
5,insertion,dvf,0.014605,0.121153
6,insertion,influence,0.204202,0.025157
7,insertion,sif,0.076922,0.110609
8,insertion,sif_plus,0.039853,0.120006
9,insertion,tracin,0.014397,0.121345


,model,metric,method_a,method_b,wilcoxon_stat,wilcoxon_p,cliffs_delta,n_pairs,wilcoxon_p_adj_holm
0,VAE,deletion_auc_norm_minus_random,dvf,influence,0.0,0.0625,-1.00,5,0.625
16,VAE,deletion_auc_norm_minus_random,influence,sif,0.0,0.0625,1.00,5,0.625
20,VAE,deletion_auc_norm_minus_random,influence,sif_plus,0.0,0.0625,1.00,5,0.625
24,VAE,deletion_auc_norm_minus_random,influence,tracin,0.0,0.0625,1.00,5,0.625
4,VAE,deletion_auc_norm_minus_random,dvf,sif,6.0,0.8125,-0.04,5,1.000
8,VAE,deletion_auc_norm_minus_random,dvf,sif_plus,7.0,1.0000,-0.20,5,1.000
12,VAE,deletion_auc_norm_minus_random,dvf,tracin,0.0,1.0000,0.04,5,1.000
28,VAE,deletion_auc_norm_minus_random,sif,sif_plus,7.0,1.0000,-0.04,5,1.000
32,VAE,deletion_auc_norm_minus_random,sif,tracin,7.0,1.0000,0.12,5,1.000
36,VAE,deletion_auc_norm_minus_random,sif_plus,tracin,7.0,1.0000,0.20,5,1.000


In [14]:
from scipy.stats import wilcoxon

def _bootstrap_ci_mean(x, n_boot=2000, alpha=0.05, seed=2026):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = []
    n = x.size
    for _ in range(int(n_boot)):
        idx = rng.integers(0, n, size=n)
        means.append(float(np.mean(x[idx])))
    lo = float(np.quantile(means, alpha / 2.0))
    hi = float(np.quantile(means, 1.0 - alpha / 2.0))
    return lo, hi

def sif_vs_tracin_proof(detail_df_in):
    if len(detail_df_in) == 0:
        return pd.DataFrame()

    metrics = [
        "deletion_auc_minus_random",
        "insertion_auc_minus_random",
        "pos_at_10",
        "ndcg_ke_auc",
    ]
    rows = []

    for model_name in sorted(detail_df_in["model"].unique().tolist()):
        dm = detail_df_in[detail_df_in["model"] == model_name].copy()
        ds = dm[dm["method"] == "sif"].sort_values("seed")
        dt = dm[dm["method"] == "tracin"].sort_values("seed")
        shared = sorted(set(ds["seed"].tolist()) & set(dt["seed"].tolist()))
        if len(shared) < 2:
            continue

        a = ds[ds["seed"].isin(shared)].sort_values("seed")
        b = dt[dt["seed"].isin(shared)].sort_values("seed")

        for met in metrics:
            xa = a[met].to_numpy(dtype=float)
            xb = b[met].to_numpy(dtype=float)
            diff = xa - xb
            n = int(diff.size)
            wins = int(np.sum(diff > 0))
            ties = int(np.sum(diff == 0))
            loss = int(np.sum(diff < 0))
            win_rate = float(wins / max(n - ties, 1)) if (n - ties) > 0 else float("nan")
            mean_diff = float(np.mean(diff)) if n > 0 else float("nan")
            lo, hi = _bootstrap_ci_mean(diff)

            try:
                w = wilcoxon(xa, xb, zero_method="wilcox", alternative="greater")
                p_one_sided = float(w.pvalue)
                stat = float(w.statistic)
            except Exception:
                p_one_sided = float("nan")
                stat = float("nan")

            rows.append({
                "model": model_name,
                "metric": met,
                "n_pairs": n,
                "wins": wins,
                "ties": ties,
                "losses": loss,
                "win_rate_excl_ties": win_rate,
                "mean_diff_sif_minus_tracin": mean_diff,
                "boot95_lo": lo,
                "boot95_hi": hi,
                "wilcoxon_stat_greater": stat,
                "wilcoxon_p_one_sided": p_one_sided,
                "supports_sif_gt_tracin": bool((not np.isnan(p_one_sided)) and (p_one_sided < 0.05) and (mean_diff > 0)),
            })

    return pd.DataFrame(rows)

proof_df = sif_vs_tracin_proof(detail_df)
proof_path = BENCH_DIR / "benchmark_sif_vs_tracin_proof.csv"
save_table(proof_df, str(proof_path))
print("Saved proof table:", proof_path)
proof_df.sort_values(["metric", "model"])

saved: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_sif_vs_tracin_proof.csv
Saved proof table: D:\挑战杯\RecAcc\RecAcc\log\notebook_runs\step5_unified_benchmark\benchmark_sif_vs_tracin_proof.csv


,model,metric,n_pairs,wins,ties,losses,win_rate_excl_ties,mean_diff_sif_minus_tracin,boot95_lo,boot95_hi,wilcoxon_stat_greater,wilcoxon_p_one_sided,supports_sif_gt_tracin
0,VAE,deletion_auc_minus_random,5,3,0,2,0.600000,-0.000627,-0.012813,0.010974,7.0,0.59375,False
1,VAE,insertion_auc_minus_random,5,5,0,0,1.000000,0.061755,0.040343,0.076232,15.0,0.03125,True
3,VAE,ndcg_ke_auc,5,2,0,3,0.400000,-0.146328,-1.946120,1.650834,7.0,0.59375,False
2,VAE,pos_at_10,5,1,2,2,0.333333,-0.040000,-0.120500,0.040000,1.5,0.87500,False


## SIF 优于 TracIn 的定向检验

> 目标假设：在相同 seed 与模型下，SIF 在关键指标上优于 TracIn。

> 统计检验：
- 单侧 Wilcoxon signed-rank（H1: SIF - TracIn > 0）
- 胜率（SIF > TracIn 的配对比例）
- 均值差与 bootstrap 95% CI

## 下一步扩展建议

1. 加入弱基线（popularity / cosine / jaccard）的显式实现与统一对照。
2. 加入 bootstrap 用户子集协议，输出鲁棒区间。
3. 加入输入扰动一致性评测（Kendall / Spearman）。
4. 在同一反事实探测 API 下补齐 NEG@K 与 Rank@K。